# Granos de café — entrenamiento en Google Colab

Entrena y compara los dos enfoques con el dataset re-exportado de Label Studio:

- **A. Una etapa**: YOLO de 14 clases (línea base).
- **B. Dos etapas**: detector de "grano" + clasificador de defectos.

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (T4 o mejor)*.
2. Sube a Google Drive el dataset preparado con `preparar_export.py` comprimido en zip,
   por ejemplo `MiDrive/granos/dataset_v2.zip`. El zip debe contener `images/`, `labels/` y `data.yaml`.

Los pesos y resultados se guardan en `MiDrive/granos/`, así no se pierden si Colab se desconecta.

## 1. Configuración

In [ ]:
# --- Ajusta estas rutas ---
DRIVE_DIR   = '/content/drive/MyDrive/granos'          # carpeta de trabajo en Drive
DATASET_ZIP = f'{DRIVE_DIR}/dataset_v2.zip'            # dataset preparado con preparar_export.py
RAMA        = 'claude/yolo-grain-detection-kqf5bz'
REPO        = 'https://github.com/luisfcollazos/luisfcollazos.github.io.git'

IMGSZ_DET = 1280   # resolución para los detectores
VENTANA   = 128    # recorte fijo (px) del clasificador
WORKERS   = 2      # procesos de carga de datos: Colab tiene 2 CPU; con más el entrenamiento puede trabarse
BATCH_DET = 4      # lote de los detectores: con ~100 granos por imagen, más agota la memoria de la GPU

In [ ]:
import subprocess
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True, check=True).stdout)
except (FileNotFoundError, subprocess.CalledProcessError):
    raise SystemExit('No hay GPU. Ve a "Entorno de ejecución → Cambiar tipo de entorno de ejecución", '
                     'elige una GPU (T4) y vuelve a ejecutar desde esta celda. Si no aparece la opción, '
                     'agotaste el cupo gratuito de GPU: espera unas horas o usa Colab Pro.')
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
RUNS = f'{DRIVE_DIR}/runs'     # pesos y gráficas de los entrenamientos (persisten en Drive)

## 2. Código y dependencias

Clona los scripts desde GitHub. Si el repositorio es privado, sube la carpeta `yolo-granos/`
a Drive y usa la segunda opción de la celda.

In [ ]:
%cd /content
!rm -rf repo && git clone -q --depth 1 -b {RAMA} {REPO} repo
# Alternativa si el repo es privado:  !cp -r {DRIVE_DIR}/yolo-granos /content/repo/
%cd /content/repo/yolo-granos
!pip -q install -r requirements.txt
import ultralytics; ultralytics.checks()

## 3. Dataset

Se copia del zip de Drive al disco local de Colab, que es mucho más rápido que leer de Drive durante el
entrenamiento. Luego se corrige `path` en el `data.yaml`, porque el original apunta a una carpeta de tu computador.

In [ ]:
!rm -rf /content/dataset_v2 && mkdir -p /content/dataset_v2
!unzip -q "{DATASET_ZIP}" -d /content/dataset_v2
# si el zip traía una carpeta dentro, súbela un nivel
import glob, shutil, yaml
if not os.path.exists('/content/dataset_v2/data.yaml'):
    interno = os.path.dirname(glob.glob('/content/dataset_v2/*/data.yaml')[0])
    for x in os.listdir(interno): shutil.move(os.path.join(interno, x), '/content/dataset_v2')

DATA = '/content/dataset_v2/data.yaml'
d = yaml.safe_load(open(DATA))
d['path'] = '/content/dataset_v2'
yaml.safe_dump(d, open(DATA, 'w'), allow_unicode=True, sort_keys=False)
print(open(DATA).read())
!python limpiar_caches.py --dataset /content/dataset_v2 --borrar

### Cómo se lanzan los entrenamientos

Cada entrenamiento corre **en segundo plano** y escribe su salida en `MiDrive/granos/logs/<nombre>.log`.
La celda muestra cada minuto un resumen corto: última época, métricas y GPU.

Esto evita un problema de Colab: con salidas muy largas (la barra de progreso se reescribe cientos de veces
por época), Colab deja de refrescar la celda y **parece** que el entrenamiento se detuvo aunque sigue corriendo.

- Si detienes la celda (■), **el entrenamiento sigue**. Para volver a verlo, ejecuta `seguir('<nombre>')`.
- Para detenerlo de verdad, ejecuta `detener('<nombre>')`.

In [ ]:
import os, subprocess, time
from IPython.display import clear_output

CODIGO = '/content/repo/yolo-granos'   # carpeta de los scripts
LOGS = f'{DRIVE_DIR}/logs'
os.makedirs(LOGS, exist_ok=True)
_procesos = {}

def _ultimas_lineas(ruta, n=12):
    try:
        with open(ruta, 'rb') as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - 20000))
            texto = f.read().decode('utf-8', 'ignore')
    except FileNotFoundError:
        return []
    # la barra de progreso usa \r: quedarse con el último estado de cada línea
    lineas = [l.split('\r')[-1].strip() for l in texto.split('\n')]
    return [l for l in lineas if l][-n:]

def _gpu():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                            '--format=csv,noheader'], capture_output=True, text=True)
        return r.stdout.strip()
    except FileNotFoundError:
        return 'sin GPU'

def entrenar(nombre, comando, cada=60):
    """Lanza `comando` en segundo plano con su salida en LOGS/<nombre>.log y lo sigue."""
    log = f'{LOGS}/{nombre}.log'
    if os.path.exists(log):  # el log anterior se guarda con otro nombre
        os.replace(log, log.replace('.log', f'_{int(time.time())}.log'))
    _procesos[nombre] = subprocess.Popen(f'cd {CODIGO} && {comando} > "{log}" 2>&1',
                                         shell=True)
    seguir(nombre, cada)

def seguir(nombre, cada=60):
    log = f'{LOGS}/{nombre}.log'
    p = _procesos.get(nombre)
    inicio = time.time()
    while True:
        clear_output(wait=True)
        vivo = p is None or p.poll() is None
        print(f'[{nombre}]  {"en curso" if vivo else "terminado (código %s)" % p.returncode}'
              f'  ·  {(time.time() - inicio) / 60:.0f} min mirando  ·  GPU: {_gpu()}')
        print('\n'.join(_ultimas_lineas(log)))
        if not vivo:
            break
        time.sleep(cada)

def detener(nombre):
    p = _procesos.get(nombre)
    if p and p.poll() is None:
        subprocess.run(f'pkill -INT -P {p.pid}', shell=True)
        print(f'Señal de parada enviada a {nombre}')

### Verificación rápida (opcional)

In [ ]:
!python 0_auditar_dataset.py --data {DATA} --grande sanog --pequeno sanop --no-medir --out {DRIVE_DIR}/auditoria

## 4. A — Una etapa (14 clases)

Los detectores usan `--batch 4`: con ~100 granos por imagen, un lote mayor (o el automático `-1`) agota la memoria de la GPU al asignar etiquetas y el entrenamiento se vuelve lentísimo, como si se bloqueara. Con una GPU de más de 24 GB puedes subirlo a 8.
150 épocas tardan aproximadamente 1,5–3 h en una T4.

In [ ]:
entrenar('completo', f'python 3_entrenar.py completo --data {DATA} --imgsz {IMGSZ_DET} --batch {BATCH_DET} --workers {WORKERS} --project {RUNS}/detect')

## 5. B — Dos etapas

In [ ]:
# recortes para el clasificador + data.yaml de una sola clase para el detector
!python 2_recortar_granos.py --data {DATA} --out /content/dataset_cls --ventana {VENTANA} \
    --balancear 600 --max-por-clase 3000 --yaml-detector /content/data_1clase.yaml

In [ ]:
entrenar('detector', f'python 3_entrenar.py detector --data /content/data_1clase.yaml --imgsz {IMGSZ_DET} --batch {BATCH_DET} --workers {WORKERS} --project {RUNS}/detect')

In [ ]:
entrenar('clasificador', f'python 3_entrenar.py clasificador --data /content/dataset_cls --batch 64 --workers {WORKERS} --project {RUNS}/classify')

## 6. Comparación en el split de test

Ninguno de los dos modelos usó el split de test para elegir pesos.
Si un nombre de carpeta cambió (por ejemplo `det_granos2`, porque entrenaste dos veces), ajústalo aquí.

In [ ]:
COMPLETO = f'{RUNS}/detect/completo_granos/weights/best.pt'
DET      = f'{RUNS}/detect/det_granos/weights/best.pt'
CLS      = f'{RUNS}/classify/cls_granos/weights/best.pt'

!python 1_diagnostico_val.py --model {COMPLETO} --data {DATA} --split test --imgsz {IMGSZ_DET}

El clasificador se entrena con clases balanceadas y por eso le quita granos a sanog. `--ajuste-prior`
corrige hacia la frecuencia real. Primero se elige TAU en **val** y después se evalúa **test** una sola vez
con ese valor (elegirlo mirando test inflaría el resultado).

In [ ]:
# 1) elegir TAU en val
!python 4_dos_etapas.py evaluar --det {DET} --cls {CLS} --data {DATA} --split val \
    --imgsz-det {IMGSZ_DET} --ventana {VENTANA} --cls-data /content/dataset_cls \
    --ajuste-prior 0,0.25,0.5,0.75,1 --out {DRIVE_DIR}/resultados_dos_etapas_val

In [ ]:
# 2) evaluar test con el TAU elegido (el de mejor recall medio por clase en val, salvo que prefieras más aciertos globales)
TAU = 0.5
!python 4_dos_etapas.py evaluar --det {DET} --cls {CLS} --data {DATA} --split test \
    --imgsz-det {IMGSZ_DET} --ventana {VENTANA} --cls-data /content/dataset_cls \
    --ajuste-prior {TAU} --out {DRIVE_DIR}/resultados_dos_etapas

In [ ]:
# guardar en Drive las matrices del diagnóstico de una etapa
!cp -r runs/detect/diag_* {DRIVE_DIR}/ 2>/dev/null; ls {DRIVE_DIR}

## Si Colab se desconecta

Los pesos quedan en Drive. Vuelve a ejecutar las secciones 1–3 y luego retoma el entrenamiento desde su `last.pt`:

In [ ]:
# ejemplo: retomar el modelo de una etapa
entrenar('completo', f'python 3_entrenar.py completo --resume {RUNS}/detect/completo_granos/weights/last.pt')